In [ ]:
%pip install mlflow
%pip install --upgrade jinja2
%pip install --upgrade Flask
%pip install setuptools
%pip install xgboost

In [2]:
import mlflow

In [3]:
from mlflow import MlflowClient
from pprint import pprint
from sklearn.ensemble import RandomForestRegressor


#### **Analyse exploratoire de données**

In [8]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Charger les données (remplace par le vrai nom de ton fichier)
df = pd.read_csv('data/Loan_Data.csv')

# Afficher les 5 premières lignes
df.head()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0


In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['customer_id','default']) # On enlève la cible et l'ID
y = df['default'] # La cible est la colonne 'default'


# Split du dataset (On garde 20% des données pour le test final)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Nombre de clients pour l'entraînement : {len(X_train)}")
print(f"Nombre de clients pour le test : {len(X_test)}")


Nombre de clients pour l'entraînement : 8000
Nombre de clients pour le test : 2000


In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 2. "Apprendre" la moyenne et l'écart-type sur le train ET transformer
X_train_scaled = scaler.fit_transform(X_train)

# 3. Transformer le test avec les mêmes paramètres (sans ré-apprendre !)
X_test_scaled = scaler.transform(X_test)

#### **Création de l'expérience Logistic_Regression**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, roc_auc_score, f1_score, confusion_matrix
import os
import numpy as np 
from sklearn.linear_model import LogisticRegression


In [10]:
client = MlflowClient(tracking_uri="http://localhost:8080")

In [11]:

experiment_description = (
   "Évaluation des modèles pour la prédiction du défaut de paiement sur des prêts personnels bancaires."
)


experiment_tags = {# tags servant à l'étiquettage et filtrage de nos expériences
    "project_name": "Pret_Bancaire_MLOps",
    "team_sector": "Banque_Detail",  
    "mlops_stage": "Model_Engineering",
    "mlflow.note.content": "experiment_description"
}


logistic_default = client.create_experiment(  # Nom de l'objet servant de connexion avec le dossier "Default_models" sur MLflow
    name="Logistic_Regression",  # Nom du dossier sur MLflow
    tags=experiment_tags
)

In [13]:
# Set the global tracking URI for MLflow

mlflow.set_tracking_uri("http://localhost:8080")

In [25]:

logistic_experiment = mlflow.set_experiment("Logistic_Regression")  # Nom du dossier sur MLflow

# If this is not set, a unique name will be auto-generated for your run.
run_name = "Logistic_Regression_v3"

# Define an artifact path that the model will be saved to.
artifact_path = "lr_v3"

params =  {"C": 0.5,       # Calculer l'ordonnée à l'origine (constante)
        "max_iter": 1000,              # Copier les données pour éviter de modifier l'original
        "solver": "lbfgs"  
}


lr = LogisticRegression(**params)
lr.fit(X_train_scaled, y_train)        
y_pred = lr.predict(X_test_scaled)
y_probs = lr.predict_proba(X_test_scaled)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_probs) 
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)          
print(f"Accuracy: {accuracy}")
print(f"ROC AUC Score: {roc_auc}")
print(f"F1 Score: {f1}")
print("Confusion Matrix:")    
print(conf_matrix)

metrics = {"accuracy" : accuracy, "roc_auc": roc_auc, "f1": f1}   

if mlflow.active_run():
    mlflow.end_run()  # Terminer la session si erreur

with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params)  # Log des hyperparamètres
        mlflow.log_metrics(metrics)  # Log des métriques
        mlflow.sklearn.log_model(sk_model=lr, input_example=X_test, artifact_path=artifact_path)  # Log du modèle



Accuracy: 0.995
ROC AUC Score: 0.9999495560936239
F1 Score: 0.9855072463768116
Confusion Matrix:
[[1650    2]
 [   8  340]]


2026/04/10 23:57:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 23:57:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\Utilisateur\Desktop\Panthéon-Sorbonne VII\Projet_MLOps\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that 

🏃 View run Logistic_Regression_v3 at: http://localhost:8080/#/experiments/1/runs/2668721e71f44cbf99150ad19419e674
🧪 View experiment at: http://localhost:8080/#/experiments/1
